# Music ML v4 — Structure-Aware Piano Generation

A ground-up rewrite of the generator (see repo-root **IMPROVEMENTS.md** for the full rationale).
The changes that turn *"a mesh of notes"* into real pieces:

1. **REMI-style tokenizer** — explicit `Bar`/`Position` metrical grid, note *durations* (no separate
   NOTE_OFF the model forgets), and **sustain-pedal**-aware note lengths.
2. **Long context (up to 2048 tokens ≈ 30–60 s) + RoPE** so attention sees whole phrases and their return.
3. **Pitch-transposition augmentation**, **per-piece** train/val split, cosine LR, label smoothing.
4. **Kaggle-safe training**: mid-epoch checkpoints, resume across sessions, wall-clock time budget, per-epoch previews.

Self-contained — does **not** use the old Phase-1/1.5 model. Run top to bottom.


## Step 1 — Locate the project + composer map, copy `src/` into the working dir


In [ ]:
import os, sys, shutil

WORK_DIR = '/kaggle/working/music_ml'
os.makedirs(WORK_DIR, exist_ok=True)

INPUT_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if root.replace('/kaggle/input','').count(os.sep) > 5:
        dirs[:] = []; continue
    if os.path.isdir(os.path.join(root,'src')) and os.path.isdir(os.path.join(root,'scripts')):
        INPUT_DIR = root; break
if INPUT_DIR is None:
    raise FileNotFoundError('Attach the music-ml project dataset (must contain src/ and scripts/).')

COMPOSER_MAP_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if root.replace('/kaggle/input','').count(os.sep) > 7:
        dirs[:] = []; continue
    if 'composer_map.json' in files:
        COMPOSER_MAP_PATH = os.path.join(root,'composer_map.json'); break
if COMPOSER_MAP_PATH is None:
    raise FileNotFoundError('composer_map.json not found under /kaggle/input.')

dst = os.path.join(WORK_DIR,'src')
if os.path.isdir(dst): shutil.rmtree(dst)
shutil.copytree(os.path.join(INPUT_DIR,'src'), dst)
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print('Project    :', INPUT_DIR)
print('ComposerMap:', COMPOSER_MAP_PATH)
print('Working dir:', os.getcwd())


## Step 2 — Install `mido`, verify GPU, auto-pick a size preset for the VRAM

Presets keep the model within the detected GPU. Override `PRESET` by hand if you want a bigger/smaller run.


In [ ]:
!pip install -q mido
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — set Accelerator to GPU (T4/P100) in the sidebar and restart.')
device = torch.device('cuda')
vram = round(torch.cuda.get_device_properties(0).total_memory/1e9, 1)
print('GPU:', torch.cuda.get_device_name(0), '|', vram, 'GB')

# ---- VRAM-aware preset (model size + context + batching) ----
if vram >= 24:      # A100 / L4 / big
    PRESET = dict(d_model=640, n_layers=10, n_heads=10, d_ff=2560, seq_len=2048, batch_size=3, grad_accum=6)
elif vram >= 14:    # T4 16GB / P100 16GB (default Kaggle)
    PRESET = dict(d_model=512, n_layers=8,  n_heads=8,  d_ff=2048, seq_len=1024, batch_size=2, grad_accum=8)
else:               # smaller cards
    PRESET = dict(d_model=384, n_layers=6,  n_heads=6,  d_ff=1536, seq_len=768,  batch_size=2, grad_accum=8)
print('Preset:', PRESET)


## Step 3 — Build (MIDI → composer) pairs and tokenize the corpus (v4)

Re-tokenizes **raw MIDI** to v4 token `.npy`, cached in `/kaggle/working/v4_tokens` (skips already-cached files,
so re-runs are fast). MAESTRO is labelled from its metadata CSV; GiantMIDI by the `lastname-firstname` filename slug.
Set the `*_DIR` variables if auto-detect misses your attached datasets.


In [ ]:
import json, csv, glob, re
from src.v4.tokenizer import REMITokenizer
from src.v4.data import tokenize_corpus

MAESTRO_DIR   = None    # dir with maestro-v*.csv + .midi files
GIANTMIDI_DIR = None    # dir of GiantMIDI *.mid files
TOKEN_DIR     = '/kaggle/working/v4_tokens'

with open(COMPOSER_MAP_PATH) as f:
    _raw = json.load(f)
composer_map = {n:i for i,n in enumerate(_raw.keys())}   # contiguous ids
tokenizer = REMITokenizer(composer_map)
print('Composers:', len(composer_map), '| v4 vocab_size:', tokenizer.vocab_size)

pairs = []

def find_maestro_csv():
    if MAESTRO_DIR:
        c = glob.glob(os.path.join(MAESTRO_DIR,'maestro-v*.csv'))
        if c: return c[0]
    hits = glob.glob('/kaggle/input/**/maestro-v*.csv', recursive=True)
    return hits[0] if hits else None

mcsv = find_maestro_csv()
if mcsv:
    base = os.path.dirname(mcsv)
    with open(mcsv) as f:
        for row in csv.DictReader(f):
            comp = row.get('canonical_composer','').strip()
            rel  = row.get('midi_filename','').strip()
            if comp in composer_map and rel and os.path.exists(os.path.join(base, rel)):
                stem = re.sub(r'[^0-9A-Za-z]+','_', os.path.splitext(os.path.basename(rel))[0])
                pairs.append((os.path.join(base, rel), composer_map[comp], f'ms_{composer_map[comp]}_{stem}'))
    print('MAESTRO pairs:', len(pairs))
else:
    print('MAESTRO csv not found — skipping.')

def slug(name):
    p = name.lower().split()
    return '-'.join([p[-1]]+p[:-1]) if len(p)>1 else p[0]
slug_map = {}
for n in composer_map:
    s = slug(n); slug_map[s] = n
    parts = s.split('-')
    if len(parts) > 2: slug_map['-'.join(parts[:2])] = n

gm_count = 0
for root in ([GIANTMIDI_DIR] if GIANTMIDI_DIR else ['/kaggle/input']):
    if not root or not os.path.isdir(root): continue
    for r, d, fs in os.walk(root):
        if r.replace('/kaggle/input','').count(os.sep) > 6: d[:]=[]; continue
        if 'maestro' in r.lower(): continue
        for fn in fs:
            if not fn.lower().endswith(('.mid','.midi')): continue
            stem = os.path.splitext(fn)[0]
            for s in sorted(slug_map, key=len, reverse=True):
                if stem.lower().startswith(s+'-'):
                    cid = composer_map[slug_map[s]]
                    out = 'gm_%d_%s' % (cid, re.sub(r'[^0-9A-Za-z]+','_', stem))
                    pairs.append((os.path.join(r,fn), cid, out)); gm_count += 1
                    break
print('GiantMIDI pairs:', gm_count, '| total pairs:', len(pairs))

assert pairs, 'No MIDI matched. Set MAESTRO_DIR / GIANTMIDI_DIR manually.'
token_files = tokenize_corpus(pairs, tokenizer, TOKEN_DIR)
print('Cached token files:', len(token_files))


## Step 4 — Dataset health check (token-length distribution + composer coverage)


In [ ]:
import numpy as np
from collections import Counter

lengths, per_comp = [], Counter()
cbase, cend = tokenizer.COMPOSER_BASE, tokenizer.COMPOSER_BASE + tokenizer.num_composers
inv_map = {v:k for k,v in composer_map.items()}
for f in token_files:
    t = np.load(f)
    lengths.append(len(t))
    if len(t) > 1 and cbase <= t[1] < cend:
        per_comp[inv_map.get(int(t[1]-cbase), '?')] += 1
lengths = np.array(lengths)
tot = int(lengths.sum())
print(f'Pieces           : {len(lengths)}')
print(f'Total tokens     : {tot:,}  (~{tot/ (60*40):,.0f} min of music at ~40 tok/s)')
print(f'Tokens/piece     : min {lengths.min()}  median {int(np.median(lengths))}  '
      f'mean {int(lengths.mean())}  max {lengths.max()}')
print(f'Composers covered: {len(per_comp)} / {len(composer_map)}')
print('Top 12 by piece count:')
for name, c in per_comp.most_common(12):
    print(f'   {c:>4d}  {name}')


## Step 5 — Round-trip sanity check (confirm encode→decode is faithful)


In [ ]:
sample = token_files[0]
toks = np.load(sample).tolist()
n = tokenizer.detokenize(toks, '/kaggle/working/roundtrip_check.mid')
print('File   :', os.path.basename(sample))
print('Tokens :', len(toks), '| Notes:', n, '-> /kaggle/working/roundtrip_check.mid')
print('First tokens:', toks[:14])


## Step 6 — Build per-piece datasets (with augmentation) + the RoPE model

Transposition augmentation (`transpose_range`) multiplies effective data and teaches key-invariance.
Train/val split is **by piece**, so no validation music leaks into training.


In [ ]:
from src.v4.data import V4Dataset, split_files
from src.v4.model import MusicTransformerV4, ModelConfig
from src.v4.train import TrainConfigV4, TrainerV4

SEQ_LEN = PRESET['seq_len']

train_files, val_files = split_files(token_files, val_frac=0.05, seed=1234)
train_ds = V4Dataset(train_files, tokenizer, seq_len=SEQ_LEN, transpose_range=(-3, 3))
val_ds   = V4Dataset(val_files,   tokenizer, seq_len=SEQ_LEN, transpose_range=(0, 0))   # no aug in val
print('pieces  train/val :', len(train_files), '/', len(val_files))
print('windows train/val :', len(train_ds), '/', len(val_ds))

mcfg = ModelConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=PRESET['d_model'], n_heads=PRESET['n_heads'],
    n_layers=PRESET['n_layers'], d_ff=PRESET['d_ff'],
    dropout=0.1, max_seq_len=SEQ_LEN, pad_id=tokenizer.PAD,
)
model = MusicTransformerV4(mcfg)
print('model params: %.1fM' % (model.num_params()/1e6))


## Step 7 — *(optional)* 60-second smoke test before the real run

Trains a couple hundred steps on a small subset to confirm the loss falls and nothing crashes. Skip once you trust the run.


In [ ]:
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    _sub = V4Dataset(train_files[:20], tokenizer, seq_len=min(SEQ_LEN, 512), transpose_range=(-3,3))
    _sm  = MusicTransformerV4(ModelConfig(
        vocab_size=tokenizer.vocab_size, d_model=256, n_heads=4, n_layers=4,
        d_ff=1024, max_seq_len=512, pad_id=tokenizer.PAD))
    _cfg = TrainConfigV4(seq_len=min(SEQ_LEN,512), batch_size=2, grad_accum=2,
                         num_epochs=1, warmup_steps=10, num_workers=2,
                         log_every=25, save_every_steps=0, ckpt_dir='/kaggle/working/_smoke')
    _t = TrainerV4(_sm, _sm.cfg, _sub, _sub, _cfg, device, composer_map)
    _t.train()
    del _sm, _t, _sub
    torch.cuda.empty_cache()
    print('Smoke test done — loss should have dropped from ~5.6.')


## Step 8 — Train  (mixed precision · cosine LR · resume-safe · time-budgeted · per-epoch previews)

`TIME_BUDGET_H` stops cleanly before Kaggle's session cap so the best checkpoint survives. Re-run this cell in a new
session to resume from `v4_last.pt`. A short preview MIDI is written every `PREVIEW_EVERY` epochs.


In [ ]:
from src.v4.generate import generate

TIME_BUDGET_H  = 8.0            # stop after the epoch that crosses this wall-clock budget
PREVIEW_EVERY  = 2             # write a preview MIDI every N epochs
PREVIEW_COMPOSER = sorted(composer_map)[0]

tcfg = TrainConfigV4(
    seq_len=SEQ_LEN,
    batch_size=PRESET['batch_size'], grad_accum=PRESET['grad_accum'],
    num_epochs=60, lr=3e-4, warmup_steps=1500, label_smoothing=0.1,
    save_every_steps=400, ckpt_dir='/kaggle/working/checkpoints_v4',
)
trainer = TrainerV4(model, mcfg, train_ds, val_ds, tcfg, device, composer_map)

# Resume if a previous session left a checkpoint
start_epoch, best_val = 1, float('inf')
last = os.path.join(tcfg.ckpt_dir, 'v4_last.pt')
if os.path.exists(last):
    ck = torch.load(last, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state'])
    trainer.optimizer.load_state_dict(ck['optim_state'])
    trainer.scheduler.load_state_dict(ck['sched_state'])
    trainer.scaler.load_state_dict(ck['scaler_state'])
    start_epoch, best_val = ck['epoch'] + 1, ck['best_val']
    print(f'Resumed from epoch {ck["epoch"]} (best_val {best_val:.4f})')

def preview(epoch, tr, va, m):
    if epoch % PREVIEW_EVERY: return
    toks = generate(m, tokenizer, PREVIEW_COMPOSER, device,
                    max_bars=24, temperature=0.95, top_p=0.92, rep_penalty=1.15)
    path = f'/kaggle/working/preview_epoch{epoch:03d}.mid'
    n = tokenizer.detokenize(toks, path)
    print(f'   preview: {n} notes -> {path}')

trainer.train(start_epoch=start_epoch, best_val=best_val,
              time_budget_s=TIME_BUDGET_H*3600, on_epoch_end=preview)


## Step 9 — Generate finished pieces  (a few candidates so you can pick the best)

Loads the best checkpoint and samples several takes. Lower `temperature` / higher `rep_penalty` = tighter & more repetitive;
higher `temperature` = more adventurous. Try a few and keep the one you like.


In [ ]:
from src.v4.train import load_checkpoint
from src.v4.generate import generate

COMPOSER    = 'Frédéric Chopin'    # any key in composer_map
N_CANDIDATES = 3
SETTINGS = [
    dict(temperature=0.90, top_p=0.92, rep_penalty=1.18),
    dict(temperature=0.98, top_p=0.94, rep_penalty=1.12),
    dict(temperature=1.05, top_p=0.95, rep_penalty=1.10),
]

gmodel, gcfg, gmap, _ = load_checkpoint('/kaggle/working/checkpoints_v4/v4_best.pt', device)
gtok = REMITokenizer(gmap)
if COMPOSER not in gmap:
    print('Pick one of:', sorted(gmap)[:20], '...'); COMPOSER = sorted(gmap)[0]

for i in range(N_CANDIDATES):
    s = SETTINGS[i % len(SETTINGS)]
    toks = generate(gmodel, gtok, COMPOSER, device, max_bars=64, max_tokens=4000, seed=i, **s)
    out = f'/kaggle/working/gen_{COMPOSER.split()[-1]}_{i}.mid'
    n = gtok.detokenize(toks, out)
    print(f'[{i}] {n:>4d} notes  T={s["temperature"]}  -> {out}')


## Step 10 — *(optional)* Render a MIDI to audio to listen inside the notebook

Needs `fluidsynth` + a soundfont; if unavailable, just download the `.mid` from the output panel and play it locally.


In [ ]:
MIDI_TO_PLAY = f'/kaggle/working/gen_{COMPOSER.split()[-1]}_0.mid'
try:
    import subprocess, glob as _glob
    from IPython.display import Audio, display
    subprocess.run('apt-get -qq install -y fluidsynth >/dev/null 2>&1', shell=True)
    !pip install -q pyfluidsynth pretty_midi
    import pretty_midi
    sf = (_glob.glob('/usr/share/sounds/sf2/*.sf2') +
          _glob.glob('/usr/share/soundfonts/*.sf2'))
    pm = pretty_midi.PrettyMIDI(MIDI_TO_PLAY)
    audio = pm.fluidsynth(sf2_path=sf[0]) if sf else pm.synthesize()
    display(Audio(audio, rate=44100))
except Exception as e:
    print('Audio render unavailable (', e, ').')
    print('Download', MIDI_TO_PLAY, 'from the right-hand Output panel and play it locally.')
